# Send the latest logs/results from Drive to GitHub (no GPU needed)

Use this when a run already finished (or crashed) and Claude needs to read the logs. Select RUN_ID (blank = latest), set PUSH_RESULTS=True and provide the Colab secret `GH_TOKEN` (key icon on the left) to upload. REPO_URL blank uses DRIVE_ROOT/code.

In [ ]:
#@title 1. Cấu hình (chỉ sửa ô này)
REPO_URL   = "https://github.com/bobbibao/ocsd-thesis.git"  #@param {type:"string"}
BRANCH     = "main"                                             #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"                 #@param {type:"string"}
TIER       = "pilot"   #@param ["semantic_screen", "repair_screen", "screen", "pilot", "paper", "full"]
BACKBONE   = "sd15"    #@param ["sd15", "sd21"]
MAX_MINUTES = 600      #@param {type:"number"}
SKIP_JOBS  = ""        #@param {type:"string"}
RUN_ID     = ""        #@param {type:"string"}
PUSH_RESULTS = True    #@param {type:"boolean"}
# pilot: kiểm tra nhanh (~1-2 giờ GPU); paper: đủ số liệu cho khóa luận; full: cho bài báo (cần thêm GPU).
# Có thể bấm "Run all" nhiều lần: ảnh/độ đo đã có trên Drive sẽ được bỏ qua và chạy tiếp phần còn lại.
# SKIP_JOBS: comma-separated job prefixes to leave out, e.g. "pareto" or "pareto,E3_quickdraw_power" (GPU budget).

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
CODE = "/content/KLTN_OCSD"
if not REPO_URL:
    CODE = os.path.join(DRIVE_ROOT, "code")
elif os.path.exists(CODE):
    print("Using existing checkout:", CODE)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, CODE], check=True)
sys.path.insert(0, CODE)
from ocsd import sync
RESULTS = sync.results_for_run(DRIVE_ROOT, RUN_ID)
print("Selected results:", RESULTS)
if PUSH_RESULTS:
    sync.try_push_results(RESULTS, sync.colab_token())
else:
    print("Set PUSH_RESULTS=True to upload this run, including its manifest")